# Exercise 4: Part-of-Speech Tagging with a Hidden Markov Model (Viterbi by hand)

**Sentence (10 words):** *"The complex houses married students and teachers in the autumn."*

**Goal:** find the most likely sequence of POS tags $\hat t_{1:10}$ with the Viterbi algorithm:

$$\hat t_{1:n} = \arg\max_{t_{1:n}} \; \pi(t_1)\,b_{t_1}(w_1)\prod_{i=2}^{n} a(t_{i-1},t_i)\, b_{t_i}(w_i)$$

The sentence is a well-known **garden-path sentence**. Most readers first take *complex* as an adjective and *houses* as a plural noun ("the complex houses…") and get stuck at *married*. The grammatical reading is:

> [The complex]$_{NP}$ [houses]$_{V}$ [married students and teachers]$_{NP}$ [in the autumn]$_{PP}$
>
> (*The housing complex provides homes for married students and teachers in the autumn.*)

So the tagger has to settle three ambiguous words: **complex** (JJ/NN), **houses** (NNS/VBZ) and **married** (JJ/VBD).

**Notebook structure**

| Part | Content |
|---|---|
| A | Model parameters: tags, initial, transition and emission probabilities |
| B | Trellis: B1 Initialization → B2 Recursion (phase 1, $t=2..5$) → B3 Phase 2 (recursion, $t=6..10$) → B4 Termination → B5 Backtracking |
| C | Log-space verification, final tag sequence and exact $v_{10}$ |

Every cell value is written out in the markdown (the "by hand" work). The code cells repeat the same computation so the numbers can be checked. The final probability is computed in exact rational arithmetic (`fractions.Fraction`).

*Convention:* the final period is punctuation and is not counted as one of the 10 words. Words are lower-cased before emission lookup, so *The* and *the* share $b_{DT}(\text{the})$.

---
# Part A: Model Parameters

## A.1 Tags

The tagset is a reduced subset of the Penn Treebank tags, covering every tag a word in this sentence could plausibly take:

| # | Tag | Meaning | Words in the sentence that can take it |
|---|---|---|---|
| 1 | **DT** | Determiner | The, the |
| 2 | **JJ** | Adjective | complex, married |
| 3 | **NN** | Noun, singular | complex, autumn |
| 4 | **NNS** | Noun, plural | houses, students, teachers |
| 5 | **VBZ** | Verb, 3rd person singular present | houses |
| 6 | **VBD** | Verb, past tense | married |
| 7 | **CC** | Coordinating conjunction | and |
| 8 | **IN** | Preposition | in |

$N = 8$ hidden states, $T = 10$ observations, so the trellis has $8 \times 10 = 80$ cells.

In [1]:
from fractions import Fraction as F
import math
import pandas as pd

words = ['The', 'complex', 'houses', 'married', 'students', 'and', 'teachers', 'in', 'the', 'autumn']
tags  = ['DT', 'JJ', 'NN', 'NNS', 'VBZ', 'VBD', 'CC', 'IN']
T, N = len(words), len(tags)
print(f"T = {T} words, N = {N} tags")

T = 10 words, N = 8 tags


## A.2 Initial probabilities $\pi(s) = P(t_1 = s)$

These are set by hand to reflect English: most sentences begin with a determiner. Some start with a preposition ("In the…"), a bare noun or a past-tense verb. Few start with a conjunction.

| Tag | DT | JJ | NN | NNS | VBZ | VBD | CC | IN | **Σ** |
|---|---|---|---|---|---|---|---|---|---|
| $\pi$ | 0.45 | 0.07 | 0.10 | 0.08 | 0.05 | 0.10 | 0.05 | 0.10 | **1.00** |

In [2]:
pi = {s: F(x) for s, x in {'DT': '0.45', 'JJ': '0.07', 'NN': '0.10', 'NNS': '0.08', 'VBZ': '0.05', 'VBD': '0.10', 'CC': '0.05', 'IN': '0.10'}.items()}
assert sum(pi.values()) == 1
pd.DataFrame({"pi": {s: float(p) for s, p in pi.items()}}).T

,DT,JJ,NN,NNS,VBZ,VBD,CC,IN
pi,0.45,0.07,0.1,0.08,0.05,0.1,0.05,0.1


## A.3 Transition probabilities $a(s', s) = P(t_i = s \mid t_{i-1} = s')$

Each row is a probability distribution, so it sums to 1. The values encode ordinary English syntax:

* **DT →** usually a noun (NN 0.45) or an adjective (JJ 0.30); almost never a verb.
* **JJ →** usually a noun (NNS 0.40, NN 0.35).
* **NN →** a singular noun subject is often followed by a 3rd-person-singular verb (VBZ 0.25), a preposition (IN 0.20) or a past verb (VBD 0.15).
* **NNS →** a plural noun is often followed by VBD (0.30), IN (0.30) or CC (0.20). It is *not* followed by VBZ, because of subject–verb agreement (0.05).
* **VBZ →** the object NP begins with DT (0.30), JJ (0.25) or NNS (0.15).
* **CC →** coordinated nouns (NNS 0.30).
* **IN →** the prepositional object begins with DT (0.55).

| from \ to | DT | JJ | NN | NNS | VBZ | VBD | CC | IN | **Σ** |
|---|---|---|---|---|---|---|---|---|---|
| **DT** | 0.01 | 0.30 | 0.45 | 0.20 | 0.01 | 0.01 | 0.01 | 0.01 | 1.00 |
| **JJ** | 0.01 | 0.10 | 0.35 | 0.40 | 0.02 | 0.02 | 0.05 | 0.05 | 1.00 |
| **NN** | 0.02 | 0.03 | 0.15 | 0.10 | 0.25 | 0.15 | 0.10 | 0.20 | 1.00 |
| **NNS** | 0.02 | 0.03 | 0.05 | 0.05 | 0.05 | 0.30 | 0.20 | 0.30 | 1.00 |
| **VBZ** | 0.30 | 0.25 | 0.10 | 0.15 | 0.01 | 0.04 | 0.05 | 0.10 | 1.00 |
| **VBD** | 0.35 | 0.10 | 0.10 | 0.20 | 0.01 | 0.01 | 0.03 | 0.20 | 1.00 |
| **CC** | 0.15 | 0.20 | 0.20 | 0.30 | 0.05 | 0.05 | 0.01 | 0.04 | 1.00 |
| **IN** | 0.55 | 0.10 | 0.15 | 0.15 | 0.01 | 0.01 | 0.01 | 0.02 | 1.00 |

In [3]:
A_rows = {'DT': ['0.01', '0.30', '0.45', '0.20', '0.01', '0.01', '0.01', '0.01'], 'JJ': ['0.01', '0.10', '0.35', '0.40', '0.02', '0.02', '0.05', '0.05'], 'NN': ['0.02', '0.03', '0.15', '0.10', '0.25', '0.15', '0.10', '0.20'], 'NNS': ['0.02', '0.03', '0.05', '0.05', '0.05', '0.30', '0.20', '0.30'], 'VBZ': ['0.30', '0.25', '0.10', '0.15', '0.01', '0.04', '0.05', '0.10'], 'VBD': ['0.35', '0.10', '0.10', '0.20', '0.01', '0.01', '0.03', '0.20'], 'CC': ['0.15', '0.20', '0.20', '0.30', '0.05', '0.05', '0.01', '0.04'], 'IN': ['0.55', '0.10', '0.15', '0.15', '0.01', '0.01', '0.01', '0.02']}
A = {s: dict(zip(tags, map(F, A_rows[s]))) for s in tags}
for s in tags:
    assert sum(A[s].values()) == 1, s
pd.DataFrame({s: {t: float(A[s][t]) for t in tags} for s in tags}).T.rename_axis("from \\ to")

,DT,JJ,NN,NNS,VBZ,VBD,CC,IN
from \ to,,,,,,,,
DT,0.01,0.30,0.45,0.20,0.01,0.01,0.01,0.01
JJ,0.01,0.10,0.35,0.40,0.02,0.02,0.05,0.05
NN,0.02,0.03,0.15,0.10,0.25,0.15,0.10,0.20
NNS,0.02,0.03,0.05,0.05,0.05,0.30,0.20,0.30
VBZ,0.30,0.25,0.10,0.15,0.01,0.04,0.05,0.10
VBD,0.35,0.10,0.10,0.20,0.01,0.01,0.03,0.20
CC,0.15,0.20,0.20,0.30,0.05,0.05,0.01,0.04
IN,0.55,0.10,0.15,0.15,0.01,0.01,0.01,0.02


## A.4 Emission probabilities $b_s(w) = P(w \mid t = s)$

Each **row** is a distribution over the whole vocabulary. Only the words in this sentence are shown; the last column holds the remaining mass, which goes to every other English word, so each row sums to 1. A word with $b_s(w)=0$ cannot take tag $s$ (for example, *the* can never be VBD).

How the values were chosen:
* Closed-class words get large emission probabilities: *the* covers 60% of DT tokens, *and* 80% of CC tokens and *in* 25% of IN tokens.
* Open-class words get small values, because each noun or verb class contains thousands of words.
* For the ambiguous words, the relative sizes follow usage. *houses* as a verb (VBZ 0.004) is more frequent **within the small VBZ class** than as a share of the huge NNS class (0.002). *married* is mostly used as an adjective, "married couple" (JJ 0.012 vs VBD 0.006). *complex* is equally common as an adjective and as a noun ("housing complex"), at 0.010 each.

| tag \ word | the | complex | houses | married | students | and | teachers | autumn | in | all other words |
|---|---|---|---|---|---|---|---|---|---|---|
| **DT** | 0.60 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0.4 |
| **JJ** | 0 | 0.010 | 0 | 0.012 | 0 | 0 | 0 | 0 | 0 | 0.978 |
| **NN** | 0 | 0.010 | 0 | 0 | 0 | 0 | 0 | 0.004 | 0 | 0.986 |
| **NNS** | 0 | 0 | 0.002 | 0 | 0.006 | 0 | 0.004 | 0 | 0 | 0.988 |
| **VBZ** | 0 | 0 | 0.004 | 0 | 0 | 0 | 0 | 0 | 0 | 0.996 |
| **VBD** | 0 | 0 | 0 | 0.006 | 0 | 0 | 0 | 0 | 0 | 0.994 |
| **CC** | 0 | 0 | 0 | 0 | 0 | 0.80 | 0 | 0 | 0 | 0.2 |
| **IN** | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0.25 | 0.75 |

In [4]:
B = {'DT': {'the': '0.60'}, 'JJ': {'complex': '0.010', 'married': '0.012'}, 'NN': {'complex': '0.010', 'autumn': '0.004'}, 'NNS': {'houses': '0.002', 'students': '0.006', 'teachers': '0.004'}, 'VBZ': {'houses': '0.004'}, 'VBD': {'married': '0.006'}, 'CC': {'and': '0.80'}, 'IN': {'in': '0.25'}}
def b(s, w):
    """Emission probability b_s(w); 0 if the word cannot take tag s."""
    return F(B[s].get(w.lower(), "0"))

pd.DataFrame({s: {w: float(b(s, w)) for w in dict.fromkeys(x.lower() for x in words)} for s in tags}).T

,the,complex,houses,married,students,and,teachers,in,autumn
DT,0.6,0.00,0.000,0.000,0.000,0.0,0.000,0.00,0.000
JJ,0.0,0.01,0.000,0.012,0.000,0.0,0.000,0.00,0.000
NN,0.0,0.01,0.000,0.000,0.000,0.0,0.000,0.00,0.004
NNS,0.0,0.00,0.002,0.000,0.006,0.0,0.004,0.00,0.000
VBZ,0.0,0.00,0.004,0.000,0.000,0.0,0.000,0.00,0.000
VBD,0.0,0.00,0.000,0.006,0.000,0.0,0.000,0.00,0.000
CC,0.0,0.00,0.000,0.000,0.000,0.8,0.000,0.00,0.000
IN,0.0,0.00,0.000,0.000,0.000,0.0,0.000,0.25,0.000


---
# Part B: Trellis Calculation

**Notation.** $v_t(s)$ is the probability of the best tag path that ends in tag $s$ at word $t$. $bt_t(s)$ is the back-pointer: the previous tag on that best path.

| Stage | Formula |
|---|---|
| Initialization | $v_1(s) = \pi(s)\, b_s(w_1)$, $\;bt_1(s)=0$ |
| Recursion | $v_t(s) = \max_{s'} \big[v_{t-1}(s')\, a(s',s)\big]\, b_s(w_t)$, $\;bt_t(s) = \arg\max_{s'} v_{t-1}(s')\,a(s',s)$ |
| Termination | $P^* = \max_s v_T(s)$, $\;q_T^* = \arg\max_s v_T(s)$ |
| Backtracking | $q_{t-1}^* = bt_t(q_t^*)$ for $t = T,\dots,2$ |

**Pruning zero cells.** If $b_s(w_t)=0$, then $v_t(s)=0$ whatever the maximum is. Those cells are shown as `0`. For the non-zero cells, *every* non-zero predecessor product $v_{t-1}(s')\,a(s',s)$ is listed and the maximum is marked in **bold**. Predecessors with $v_{t-1}(s')=0$ contribute 0 and are left out.

## B.1 Initialization ($t = 1$, $w_1$ = "The")

$$v_1(s) = \pi(s)\cdot b_s(\text{"the"})$$

| $s$ | $\pi(s)$ | $b_s(\text{the})$ | $v_1(s)$ |
|---|---|---|---|
| DT | 0.45 | 0.60 | 0.45 × 0.60 = **2.7000e-01** |
| JJ | 0.07 | 0 | 0.07 × 0 = **0** |
| NN | 0.10 | 0 | 0.10 × 0 = **0** |
| NNS | 0.08 | 0 | 0.08 × 0 = **0** |
| VBZ | 0.05 | 0 | 0.05 × 0 = **0** |
| VBD | 0.10 | 0 | 0.10 × 0 = **0** |
| CC | 0.05 | 0 | 0.05 × 0 = **0** |
| IN | 0.10 | 0 | 0.10 × 0 = **0** |

Only **DT** survives: $v_1(\text{DT}) = 0.45 \times 0.60 = 0.27$. All back-pointers are $bt_1(s) = 0$ (start).

## B.2 Recursion: phase 1 (the ambiguous region, $t = 2 \dots 5$)

This phase settles the garden-path ambiguity. At each step **two competing paths survive**, and the comparisons at $t=3$, $4$ and $5$ decide between the "noun-phrase" reading (*complex/JJ houses/NNS married/VBD*) and the "verb" reading (*complex/NN houses/VBZ married/JJ*).

$$v_t(s) = \max_{s'}\big[v_{t-1}(s')\,a(s',s)\big]\cdot b_s(w_t)$$

### $t = 2$, $w_{2}$ = "complex"

**JJ**: $b_{JJ}(\text{complex}) = 0.010$

$$\begin{aligned}v_{1}(\text{DT})\,a(\text{DT},\text{JJ}) = 2.7000e-01\times0.30 = 8.1000e-02\end{aligned}$$

$v_{2}(\text{JJ}) = 8.1000e-02 \times 0.010 = \boxed{8.1000e-04}$, $\;bt_{2}(\text{JJ}) = \text{DT}$

**NN**: $b_{NN}(\text{complex}) = 0.010$

$$\begin{aligned}v_{1}(\text{DT})\,a(\text{DT},\text{NN}) = 2.7000e-01\times0.45 = 1.2150e-01\end{aligned}$$

$v_{2}(\text{NN}) = 1.2150e-01 \times 0.010 = \boxed{1.2150e-03}$, $\;bt_{2}(\text{NN}) = \text{DT}$

All other tags (DT, NNS, VBZ, VBD, CC, IN) have $b_s(\text{complex}) = 0$, so $v_{2}(s) = 0$.

### $t = 3$, $w_{3}$ = "houses"

**NNS**: $b_{NNS}(\text{houses}) = 0.002$

$$\begin{aligned}\mathbf{v_{2}(\text{JJ})\,a(\text{JJ},\text{NNS}) = 8.1000e-04\times0.40 = 3.2400e-04}\\ v_{2}(\text{NN})\,a(\text{NN},\text{NNS}) = 1.2150e-03\times0.10 = 1.2150e-04\end{aligned}$$

$v_{3}(\text{NNS}) = 3.2400e-04 \times 0.002 = \boxed{6.4800e-07}$, $\;bt_{3}(\text{NNS}) = \text{JJ}$

**VBZ**: $b_{VBZ}(\text{houses}) = 0.004$

$$\begin{aligned}v_{2}(\text{JJ})\,a(\text{JJ},\text{VBZ}) = 8.1000e-04\times0.02 = 1.6200e-05\\ \mathbf{v_{2}(\text{NN})\,a(\text{NN},\text{VBZ}) = 1.2150e-03\times0.25 = 3.0375e-04}\end{aligned}$$

$v_{3}(\text{VBZ}) = 3.0375e-04 \times 0.004 = \boxed{1.2150e-06}$, $\;bt_{3}(\text{VBZ}) = \text{NN}$

All other tags (DT, JJ, NN, VBD, CC, IN) have $b_s(\text{houses}) = 0$, so $v_{3}(s) = 0$.

### $t = 4$, $w_{4}$ = "married"

**JJ**: $b_{JJ}(\text{married}) = 0.012$

$$\begin{aligned}v_{3}(\text{NNS})\,a(\text{NNS},\text{JJ}) = 6.4800e-07\times0.03 = 1.9440e-08\\ \mathbf{v_{3}(\text{VBZ})\,a(\text{VBZ},\text{JJ}) = 1.2150e-06\times0.25 = 3.0375e-07}\end{aligned}$$

$v_{4}(\text{JJ}) = 3.0375e-07 \times 0.012 = \boxed{3.6450e-09}$, $\;bt_{4}(\text{JJ}) = \text{VBZ}$

**VBD**: $b_{VBD}(\text{married}) = 0.006$

$$\begin{aligned}\mathbf{v_{3}(\text{NNS})\,a(\text{NNS},\text{VBD}) = 6.4800e-07\times0.30 = 1.9440e-07}\\ v_{3}(\text{VBZ})\,a(\text{VBZ},\text{VBD}) = 1.2150e-06\times0.04 = 4.8600e-08\end{aligned}$$

$v_{4}(\text{VBD}) = 1.9440e-07 \times 0.006 = \boxed{1.1664e-09}$, $\;bt_{4}(\text{VBD}) = \text{NNS}$

All other tags (DT, NN, NNS, VBZ, CC, IN) have $b_s(\text{married}) = 0$, so $v_{4}(s) = 0$.

### $t = 5$, $w_{5}$ = "students"

**NNS**: $b_{NNS}(\text{students}) = 0.006$

$$\begin{aligned}\mathbf{v_{4}(\text{JJ})\,a(\text{JJ},\text{NNS}) = 3.6450e-09\times0.40 = 1.4580e-09}\\ v_{4}(\text{VBD})\,a(\text{VBD},\text{NNS}) = 1.1664e-09\times0.20 = 2.3328e-10\end{aligned}$$

$v_{5}(\text{NNS}) = 1.4580e-09 \times 0.006 = \boxed{8.7480e-12}$, $\;bt_{5}(\text{NNS}) = \text{JJ}$

All other tags (DT, JJ, NN, VBZ, VBD, CC, IN) have $b_s(\text{students}) = 0$, so $v_{5}(s) = 0$.

### Phase-1 summary: how the garden path is resolved

* $t=3$ (*houses*). Both readings survive: $v_3(\text{NNS})$ (complex/JJ → houses/NNS) = 6.4800e-07 and $v_3(\text{VBZ})$ (complex/NN → houses/VBZ) = 1.2150e-06. The NNS reading is the "garden path" a human reader takes. It is only about 2× less likely than the verb reading, so Viterbi keeps it alive.
* $t=4$ (*married*). $v_4(\text{VBD})$ continues the NP reading via NNS→VBD: 1.1664e-09. $v_4(\text{JJ})$ continues the verb reading via VBZ→JJ: 3.6450e-09.
* $t=5$ (*students*, NNS). The two paths meet: VBD→NNS gives 2.3328e-10 and JJ→NNS gives **1.4580e-09**. The verb reading wins, because $a(\text{JJ},\text{NNS})=0.40$ is much larger than $a(\text{VBD},\text{NNS})=0.20$, and *married* is more likely as an adjective. From here on there is only one live path into NNS, so the decision cannot be reversed.

## B.3 Phase 2 (remaining recursion, $t = 6 \dots 10$)

The rest of the sentence (*and teachers in the autumn*) is unambiguous under the model, since each word has one possible tag. Each step therefore has a single live cell, and the recursion multiplies one transition and one emission onto the running best path.

### $t = 6$, $w_{6}$ = "and"

**CC**: $b_{CC}(\text{and}) = 0.80$

$$\begin{aligned}v_{5}(\text{NNS})\,a(\text{NNS},\text{CC}) = 8.7480e-12\times0.20 = 1.7496e-12\end{aligned}$$

$v_{6}(\text{CC}) = 1.7496e-12 \times 0.80 = \boxed{1.3997e-12}$, $\;bt_{6}(\text{CC}) = \text{NNS}$

All other tags (DT, JJ, NN, NNS, VBZ, VBD, IN) have $b_s(\text{and}) = 0$, so $v_{6}(s) = 0$.

### $t = 7$, $w_{7}$ = "teachers"

**NNS**: $b_{NNS}(\text{teachers}) = 0.004$

$$\begin{aligned}v_{6}(\text{CC})\,a(\text{CC},\text{NNS}) = 1.3997e-12\times0.30 = 4.1990e-13\end{aligned}$$

$v_{7}(\text{NNS}) = 4.1990e-13 \times 0.004 = \boxed{1.6796e-15}$, $\;bt_{7}(\text{NNS}) = \text{CC}$

All other tags (DT, JJ, NN, VBZ, VBD, CC, IN) have $b_s(\text{teachers}) = 0$, so $v_{7}(s) = 0$.

### $t = 8$, $w_{8}$ = "in"

**IN**: $b_{IN}(\text{in}) = 0.25$

$$\begin{aligned}v_{7}(\text{NNS})\,a(\text{NNS},\text{IN}) = 1.6796e-15\times0.30 = 5.0388e-16\end{aligned}$$

$v_{8}(\text{IN}) = 5.0388e-16 \times 0.25 = \boxed{1.2597e-16}$, $\;bt_{8}(\text{IN}) = \text{NNS}$

All other tags (DT, JJ, NN, NNS, VBZ, VBD, CC) have $b_s(\text{in}) = 0$, so $v_{8}(s) = 0$.

### $t = 9$, $w_{9}$ = "the"

**DT**: $b_{DT}(\text{the}) = 0.60$

$$\begin{aligned}v_{8}(\text{IN})\,a(\text{IN},\text{DT}) = 1.2597e-16\times0.55 = 6.9284e-17\end{aligned}$$

$v_{9}(\text{DT}) = 6.9284e-17 \times 0.60 = \boxed{4.1570e-17}$, $\;bt_{9}(\text{DT}) = \text{IN}$

All other tags (JJ, NN, NNS, VBZ, VBD, CC, IN) have $b_s(\text{the}) = 0$, so $v_{9}(s) = 0$.

### $t = 10$, $w_{10}$ = "autumn"

**NN**: $b_{NN}(\text{autumn}) = 0.004$

$$\begin{aligned}v_{9}(\text{DT})\,a(\text{DT},\text{NN}) = 4.1570e-17\times0.45 = 1.8707e-17\end{aligned}$$

$v_{10}(\text{NN}) = 1.8707e-17 \times 0.004 = \boxed{7.4827e-20}$, $\;bt_{10}(\text{NN}) = \text{DT}$

All other tags (DT, JJ, NNS, VBZ, VBD, CC, IN) have $b_s(\text{autumn}) = 0$, so $v_{10}(s) = 0$.

### Complete trellis (value $v_t(s)$ and back-pointer ←$bt_t(s)$)

| $s$ \ $t$ | 1<br>*The* | 2<br>*complex* | 3<br>*houses* | 4<br>*married* | 5<br>*students* | 6<br>*and* | 7<br>*teachers* | 8<br>*in* | 9<br>*the* | 10<br>*autumn* |
|---|---|---|---|---|---|---|---|---|---|---|
| **DT** | 2.7000e-01<br>←start | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 4.1570e-17<br>←IN | 0 |
| **JJ** | 0 | 8.1000e-04<br>←DT | 0 | 3.6450e-09<br>←VBZ | 0 | 0 | 0 | 0 | 0 | 0 |
| **NN** | 0 | 1.2150e-03<br>←DT | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 7.4827e-20<br>←DT |
| **NNS** | 0 | 0 | 6.4800e-07<br>←JJ | 0 | 8.7480e-12<br>←JJ | 0 | 1.6796e-15<br>←CC | 0 | 0 | 0 |
| **VBZ** | 0 | 0 | 1.2150e-06<br>←NN | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| **VBD** | 0 | 0 | 0 | 1.1664e-09<br>←NNS | 0 | 0 | 0 | 0 | 0 | 0 |
| **CC** | 0 | 0 | 0 | 0 | 0 | 1.3997e-12<br>←NNS | 0 | 0 | 0 | 0 |
| **IN** | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1.2597e-16<br>←NNS | 0 | 0 |

In [5]:
def viterbi(words, tags, pi, A, b, verbose=True):
    T = len(words)
    v  = [{s: pi[s] * b(s, words[0]) for s in tags}]
    bt = [{s: None for s in tags}]
    if verbose:
        print(f"t=1 '{words[0]}'  (Initialization)")
        for s in tags:
            if v[0][s]: print(f"   v1({s}) = {float(pi[s])} x {float(b(s, words[0]))} = {float(v[0][s]):.4e}")
    for t in range(1, T):
        v.append({}); bt.append({})
        if verbose: print(f"\nt={t+1} '{words[t]}'")
        for s in tags:
            cand = {p: v[t-1][p] * A[p][s] for p in tags}
            best = max(cand, key=cand.get)
            v[t][s]  = cand[best] * b(s, words[t])
            bt[t][s] = best if v[t][s] > 0 else None
            if verbose and v[t][s] > 0:
                terms = ", ".join(f"{p}:{float(c):.3e}" for p, c in cand.items() if c > 0)
                print(f"   v{t+1}({s}) = max[{terms}] -> {best}  x b={float(b(s, words[t]))}  = {float(v[t][s]):.4e}")
    return v, bt

v, bt = viterbi(words, tags, pi, A, b)

t=1 'The'  (Initialization)
   v1(DT) = 0.45 x 0.6 = 2.7000e-01

t=2 'complex'
   v2(JJ) = max[DT:8.100e-02] -> DT  x b=0.01  = 8.1000e-04
   v2(NN) = max[DT:1.215e-01] -> DT  x b=0.01  = 1.2150e-03

t=3 'houses'
   v3(NNS) = max[JJ:3.240e-04, NN:1.215e-04] -> JJ  x b=0.002  = 6.4800e-07
   v3(VBZ) = max[JJ:1.620e-05, NN:3.037e-04] -> NN  x b=0.004  = 1.2150e-06

t=4 'married'
   v4(JJ) = max[NNS:1.944e-08, VBZ:3.038e-07] -> VBZ  x b=0.012  = 3.6450e-09
   v4(VBD) = max[NNS:1.944e-07, VBZ:4.860e-08] -> NNS  x b=0.006  = 1.1664e-09

t=5 'students'
   v5(NNS) = max[JJ:1.458e-09, VBD:2.333e-10] -> JJ  x b=0.006  = 8.7480e-12

t=6 'and'
   v6(CC) = max[NNS:1.750e-12] -> NNS  x b=0.8  = 1.3997e-12

t=7 'teachers'
   v7(NNS) = max[CC:4.199e-13] -> CC  x b=0.004  = 1.6796e-15

t=8 'in'
   v8(IN) = max[NNS:5.039e-16] -> NNS  x b=0.25  = 1.2597e-16

t=9 'the'
   v9(DT) = max[IN:6.928e-17] -> IN  x b=0.6  = 4.1570e-17

t=10 'autumn'
   v10(NN) = max[DT:1.871e-17] -> DT  x b=0.004  = 7.4827e-20


In [6]:
# Full trellis as a table (0 = impossible cell)
trellis = pd.DataFrame({f"{t+1}:{w}": {s: float(v[t][s]) for s in tags} for t, w in enumerate(words)})
trellis.style.format(lambda x: f"{x:.3e}" if x else "0")

,1:The,2:complex,3:houses,4:married,5:students,6:and,7:teachers,8:in,9:the,10:autumn
DT,2.700e-01,0,0,0,0,0,0,0,4.157e-17,0
JJ,0,8.100e-04,0,3.645e-09,0,0,0,0,0,0
NN,0,1.215e-03,0,0,0,0,0,0,0,7.483e-20
NNS,0,0,6.480e-07,0,8.748e-12,0,1.680e-15,0,0,0
VBZ,0,0,1.215e-06,0,0,0,0,0,0,0
VBD,0,0,0,1.166e-09,0,0,0,0,0,0
CC,0,0,0,0,0,1.400e-12,0,0,0,0
IN,0,0,0,0,0,0,0,1.260e-16,0,0


## B.4 Termination

$$P^* = \max_{s} v_{10}(s), \qquad q^*_{10} = \arg\max_s v_{10}(s)$$

At $t = 10$ ("autumn") the only non-zero cell is $v_{10}(\text{NN})$:

$$P^* = v_{10}(\text{NN}) = 7.4827e-20, \qquad q^*_{10} = \text{NN}$$

(No explicit end-of-sentence transition is used, so termination is a plain maximum over the last column.)

In [7]:
last = max(tags, key=lambda s: v[-1][s])
P_star = v[-1][last]
print("q*_10 =", last)
print("P*    =", f"{float(P_star):.6e}")

q*_10 = NN
P*    = 7.482689e-20


## B.5 Backtracking

Start from $q^*_{10} = \text{NN}$ and follow the back-pointers to the beginning:

| Step | Current tag $q^*_t$ | Back-pointer read | Previous tag $q^*_{t-1}$ |
|---|---|---|---|
| $t=10$ (autumn) | NN | $bt_{10}(\text{NN})$ | **DT** |
| $t=9$ (the) | DT | $bt_{9}(\text{DT})$ | **IN** |
| $t=8$ (in) | IN | $bt_{8}(\text{IN})$ | **NNS** |
| $t=7$ (teachers) | NNS | $bt_{7}(\text{NNS})$ | **CC** |
| $t=6$ (and) | CC | $bt_{6}(\text{CC})$ | **NNS** |
| $t=5$ (students) | NNS | $bt_{5}(\text{NNS})$ | **JJ** |
| $t=4$ (married) | JJ | $bt_{4}(\text{JJ})$ | **VBZ** |
| $t=3$ (houses) | VBZ | $bt_{3}(\text{VBZ})$ | **NN** |
| $t=2$ (complex) | NN | $bt_{2}(\text{NN})$ | **DT** |
| $t=1$ (The) | DT | $bt_1(\text{DT})$ = start | — |

**Best tag sequence:**

| The | complex | houses | married | students | and | teachers | in | the | autumn |
|---|---|---|---|---|---|---|---|---|---|
| **DT** | **NN** | **VBZ** | **JJ** | **NNS** | **CC** | **NNS** | **IN** | **DT** | **NN** |

The tagger finds the grammatical (non-garden-path) reading: *complex* is a noun, *houses* is the main verb and *married* is an adjective modifying *students*.

In [8]:
path = [last]
for t in range(len(words) - 1, 0, -1):
    path.insert(0, bt[t][path[0]])
for t, (w, s) in enumerate(zip(words, path), 1):
    print(f"t={t:2d}  {w:9s} -> {s:4s}  v={float(v[t-1][s]):.4e}")

t= 1  The       -> DT    v=2.7000e-01
t= 2  complex   -> NN    v=1.2150e-03
t= 3  houses    -> VBZ   v=1.2150e-06
t= 4  married   -> JJ    v=3.6450e-09
t= 5  students  -> NNS   v=8.7480e-12
t= 6  and       -> CC    v=1.3997e-12
t= 7  teachers  -> NNS   v=1.6796e-15
t= 8  in        -> IN    v=1.2597e-16
t= 9  the       -> DT    v=4.1570e-17
t=10  autumn    -> NN    v=7.4827e-20


---
# Part C: Log-Space Verification

Multiplying 20 small probabilities gives a number near $10^{-20}$. For longer sentences this underflows floating point, so real taggers run Viterbi in log space, where products become sums:

$$\log v_t(s) = \max_{s'}\big[\log v_{t-1}(s') + \log a(s',s)\big] + \log b_s(w_t)$$

Since $\log$ is monotonic, the arg-max (and so the tag path) is unchanged. Below, (1) the score of the chosen path is recomputed as a sum of logs, factor by factor, (2) the whole Viterbi algorithm is rerun in log space to confirm it returns the same path, and (3) the exact value of $v_{10}$ is given.

## C.1 Path score as a sum of natural logs

$$\ln v_{10} = \ln\pi(t_1) + \ln b_{t_1}(w_1) + \sum_{i=2}^{10}\big[\ln a(t_{i-1},t_i) + \ln b_{t_i}(w_i)\big]$$

| # | Factor | Probability $p$ | $\ln p$ | running $\sum \ln p$ |
|---|---|---|---|---|
| 1 | $\pi(\text{DT})$ | 0.45 | -0.798508 | -0.798508 |
| 2 | $b_{DT}(\text{the})$ | 0.60 | -0.510826 | -1.309333 |
| 3 | $a(\text{DT},\text{NN})$ | 0.45 | -0.798508 | -2.107841 |
| 4 | $b_{NN}(\text{complex})$ | 0.010 | -4.605170 | -6.713011 |
| 5 | $a(\text{NN},\text{VBZ})$ | 0.25 | -1.386294 | -8.099306 |
| 6 | $b_{VBZ}(\text{houses})$ | 0.004 | -5.521461 | -13.620766 |
| 7 | $a(\text{VBZ},\text{JJ})$ | 0.25 | -1.386294 | -15.007061 |
| 8 | $b_{JJ}(\text{married})$ | 0.012 | -4.422849 | -19.429909 |
| 9 | $a(\text{JJ},\text{NNS})$ | 0.40 | -0.916291 | -20.346200 |
| 10 | $b_{NNS}(\text{students})$ | 0.006 | -5.115996 | -25.462196 |
| 11 | $a(\text{NNS},\text{CC})$ | 0.20 | -1.609438 | -27.071634 |
| 12 | $b_{CC}(\text{and})$ | 0.80 | -0.223144 | -27.294777 |
| 13 | $a(\text{CC},\text{NNS})$ | 0.30 | -1.203973 | -28.498750 |
| 14 | $b_{NNS}(\text{teachers})$ | 0.004 | -5.521461 | -34.020211 |
| 15 | $a(\text{NNS},\text{IN})$ | 0.30 | -1.203973 | -35.224184 |
| 16 | $b_{IN}(\text{in})$ | 0.25 | -1.386294 | -36.610478 |
| 17 | $a(\text{IN},\text{DT})$ | 0.55 | -0.597837 | -37.208315 |
| 18 | $b_{DT}(\text{the})$ | 0.60 | -0.510826 | -37.719141 |
| 19 | $a(\text{DT},\text{NN})$ | 0.45 | -0.798508 | -38.517649 |
| 20 | $b_{NN}(\text{autumn})$ | 0.004 | -5.521461 | -44.039110 |

$$\ln v_{10} = -44.039110 \;\Rightarrow\; v_{10} = e^{-44.039110} = 7.4827e-20$$

Base 10: $\log_{10} v_{10} = -19.125942$, so $v_{10} = 10^{-19.125942} = 7.4827e-20$. This matches the probability-space result from B.4.

In [9]:
# C.1  sum of log factors along the chosen path
log_sum, rows = 0.0, []
for t, (w, s) in enumerate(zip(words, path)):
    trans = pi[s] if t == 0 else A[path[t-1]][s]
    emit  = b(s, w)
    for name, p in [("pi" if t == 0 else f"a({path[t-1]},{s})", trans), (f"b_{s}({w.lower()})", emit)]:
        log_sum += math.log(p)
        rows.append((name, float(p), math.log(p), log_sum))
display(pd.DataFrame(rows, columns=["factor", "p", "ln p", "running sum"]))
print(f"ln v10 = {log_sum:.6f}   ->  exp = {math.exp(log_sum):.6e}")
print(f"probability-space v10 = {float(P_star):.6e}")
assert math.isclose(math.exp(log_sum), float(P_star), rel_tol=1e-12)

,factor,p,ln p,running sum
0,pi,0.450,-0.798508,-0.798508
1,b_DT(the),0.600,-0.510826,-1.309333
2,"a(DT,NN)",0.450,-0.798508,-2.107841
3,b_NN(complex),0.010,-4.605170,-6.713011
4,"a(NN,VBZ)",0.250,-1.386294,-8.099306
5,b_VBZ(houses),0.004,-5.521461,-13.620766
6,"a(VBZ,JJ)",0.250,-1.386294,-15.007061
7,b_JJ(married),0.012,-4.422849,-19.429909
8,"a(JJ,NNS)",0.400,-0.916291,-20.346200
9,b_NNS(students),0.006,-5.115996,-25.462196


ln v10 = -44.039110   ->  exp = 7.482689e-20
probability-space v10 = 7.482689e-20


In [10]:
# C.2  full Viterbi in log space (log 0 = -inf) - must give the same path
NEG = float("-inf")
lg = lambda x: math.log(x) if x > 0 else NEG
lv  = [{s: lg(pi[s]) + lg(b(s, words[0])) for s in tags}]
lbt = [{}]
for t in range(1, len(words)):
    lv.append({}); lbt.append({})
    for s in tags:
        best = max(tags, key=lambda p: lv[t-1][p] + lg(A[p][s]))
        lv[t][s]  = lv[t-1][best] + lg(A[best][s]) + lg(b(s, words[t]))
        lbt[t][s] = best
llast = max(tags, key=lambda s: lv[-1][s])
lpath = [llast]
for t in range(len(words) - 1, 0, -1):
    lpath.insert(0, lbt[t][lpath[0]])

display(pd.DataFrame({f"{t+1}:{w}": {s: lv[t][s] for s in tags} for t, w in enumerate(words)})
        .style.format(lambda x: f"{x:.3f}" if x != NEG else "-inf"))
print("log-space path :", lpath)
print("prob-space path:", path)
print("max log v10    =", f"{lv[-1][llast]:.6f}", "-> exp =", f"{math.exp(lv[-1][llast]):.6e}")
assert lpath == path

,1:The,2:complex,3:houses,4:married,5:students,6:and,7:teachers,8:in,9:the,10:autumn
DT,-1.309,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-37.719,-inf
JJ,-inf,-7.118,-inf,-19.430,-inf,-inf,-inf,-inf,-inf,-inf
NN,-inf,-6.713,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-44.039
NNS,-inf,-inf,-14.249,-inf,-25.462,-inf,-34.020,-inf,-inf,-inf
VBZ,-inf,-inf,-13.621,-inf,-inf,-inf,-inf,-inf,-inf,-inf
VBD,-inf,-inf,-inf,-20.569,-inf,-inf,-inf,-inf,-inf,-inf
CC,-inf,-inf,-inf,-inf,-inf,-27.295,-inf,-inf,-inf,-inf
IN,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-36.610,-inf,-inf


log-space path : ['DT', 'NN', 'VBZ', 'JJ', 'NNS', 'CC', 'NNS', 'IN', 'DT', 'NN']
prob-space path: ['DT', 'NN', 'VBZ', 'JJ', 'NNS', 'CC', 'NNS', 'IN', 'DT', 'NN']
max log v10    = -44.039110 -> exp = 7.482689e-20


## C.3 Final answer

**Most likely tag sequence (10 tags):**

| The | complex | houses | married | students | and | teachers | in | the | autumn |
|---|---|---|---|---|---|---|---|---|---|
| **DT** | **NN** | **VBZ** | **JJ** | **NNS** | **CC** | **NNS** | **IN** | **DT** | **NN** |

**Exact probability of this path:**

$$v_{10} = 0.45 \times 0.60 \times 0.45 \times 0.010 \times 0.25 \times 0.004 \times 0.25 \times 0.012 \times 0.40 \times 0.006 \times 0.20 \times 0.80 \times 0.30 \times 0.004 \times 0.30 \times 0.25 \times 0.55 \times 0.60 \times 0.45 \times 0.004$$

$$v_{10} = \frac{5845851}{78125000000000000000000000} = 0.0000000000000000000748268928$$

$$\boxed{v_{10} \approx 7.48268928e-20} \qquad (\ln v_{10} = -44.039110,\;\; \log_{10} v_{10} = -19.125942)$$

The probability-space trellis, the factor-by-factor log sum and the full log-space Viterbi all give the same path and the same $v_{10}$.

In [11]:
# C.3  exact rational value of v10 (no rounding anywhere)
from decimal import Decimal, getcontext
getcontext().prec = 50
print("Final tags :", " ".join(f"{w}/{s}" for w, s in zip(words, path)))
print("v10 exact  =", P_star, "=", Decimal(P_star.numerator) / Decimal(P_star.denominator))
print("v10 approx =", f"{float(P_star):.8e}")

Final tags : The/DT complex/NN houses/VBZ married/JJ students/NNS and/CC teachers/NNS in/IN the/DT autumn/NN
v10 exact  = 5845851/78125000000000000000000000 = 7.48268928E-20
v10 approx = 7.48268928e-20


---
## Discussion

* **Why the garden path is avoided.** Up to $t=4$ the adjective/plural-noun reading is still competitive (6.4800e-07 vs 1.2150e-06 at $t=3$), and at $t=4$ it extends through $v_4(\text{VBD})$. Viterbi keeps the best path into *every* state and decides only when *students* arrives. A human reader commits greedily and has to back up. Viterbi keeps both options open and makes a globally optimal choice.
* **Sensitivity.** The result depends on the parameters. If $b_{VBZ}(\text{houses})$ were lowered from 0.004 to 0.0005 (that is, if *houses* were rarely a verb), the HMM would pick the garden-path reading DT JJ NNS VBD NNS…, an error that real HMM taggers do make on this sentence. Because a bigram HMM sees only adjacent tags, it can reach the right reading only through local statistics, not through the whole-sentence structure.
* **Why log space matters.** $v_{10}\approx 7.5\times10^{-20}$ after only 10 words. A 50-word sentence would approach the limits of double precision ($\approx10^{-308}$). Sums of logs stay in a safe numeric range and give the same arg-max.